# Step 5 — Cost / BOQ / LCOE

Replicates Annex-VI's economic model (Bill of Quantities, operating-cost present value, LCOE), with:
- the two BOQ items the methodology doc flagged as "should be a live computed dependency, not a hand-typed number" (battery system, and — a bonus fix of the same kind — solar panels) now genuinely computed from Step 4's numbers, so changing the PV size or battery size upstream updates the BOQ automatically;
- a currency toggle (PKR / EUR / USD);
- a lump-sum capital-cost override, for a quick estimate without an itemized BOQ;
- **LCOE = Total Cost / Total PV Generation**, per your correction (decision #3) — not total demand, which is what the source workbook actually (mislabeled) used.

Same approach as every step so far: validate against the workbook's own cached totals first, then apply this project's corrections and quantify the difference.

## 5.1 Setup

In [1]:
import pandas as pd  # pandas: tabular data (DataFrames)
import numpy as np  # numpy: numerical operations
from pathlib import Path  # Path: OS-independent file paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # find the project's root folder regardless of where the notebook is run from
DATA_DIR = PROJECT_ROOT / "data"  # the shared data/ folder used by every notebook

SOURCE_WORKBOOK = PROJECT_ROOT / "Base-Calculations-00.xlsx"
if not SOURCE_WORKBOOK.exists():
    SOURCE_WORKBOOK = Path("/mnt/user-data/uploads/Rural Electrification/Base-Calculations.xlsx")

# Step 4's results — this is what makes the BOQ's panel/battery lines "live" rather than hand-typed.
pv_battery_results = pd.read_csv(DATA_DIR / "pv_battery_sizing_results_2026.csv").set_index("result")["value"]
battery_capacity_kwh = pv_battery_results["battery_capacity_kwh"]
annual_egen_wh = pv_battery_results["annual_egen_wh"]
annual_demand_wh = pv_battery_results["annual_demand_wh"]

pv_parameters = pd.read_csv(DATA_DIR / "default_pv_parameters.csv").set_index("parameter")["value"]
ppeak_w = pv_parameters["ppeak_w"]

print(f"From Step 4 — battery: {battery_capacity_kwh:,.0f} kWh, Ppeak: {ppeak_w:,.0f} W, annual generation: {annual_egen_wh:,.0f} Wh")

From Step 4 — battery: 8,000 kWh, Ppeak: 2,000,000 W, annual generation: 3,839,823,405 Wh


## 5.2 Bill of Quantities (BOQ)

**A finding worth flagging plainly**: in the source workbook, only 3 of the 17 capital-cost line items actually have a cost attached — Solar Panels (164,000,000 PKR), the Battery System (400,000,000 PKR), and a lump "Cables & distribution infrastructure" (40,000,000 PKR). The other 14 (inverters, DC/AC cabling, grounding, breakers, switchgear, cable trays, mounting structure, civil work, water cleaning, transport, commissioning) have quantities filled in but **zero cost** — they simply don't add anything to the 604,000,000 PKR grand total, which is really just `164,000,000 + 400,000,000 + 40,000,000`. This isn't a deliberate design choice, it looks like an unfinished BOQ. It's reproduced faithfully below (so the validation step matches the workbook exactly), but every row is a normal editable `total_cost_pkr` value — filling in the other 14 items' real costs is just a matter of editing the table.

The **Solar Panels** and **Battery System** rows are the two the methodology doc flagged as "hand-typed, should be a live computed dependency" — here they're computed from Step 4's `ppeak_w` and `battery_capacity_kwh` each time this runs, not fixed numbers.

In [2]:
# Every BOQ line item's cost, as an editable total (matches the source workbook's own numbers, including its 14 zero-cost rows).
# "Solar Panels" and "Battery System" are placeholders here — compute_boq_total() below overrides them with a live calculation.
default_boq_items = pd.DataFrame([
    {"item_no": 1, "description": "Solar Panels (590 W rated)", "unit": "W", "qty_reference": 2_000_000, "total_cost_pkr": 164_000_000},
    {"item_no": 2, "description": "Inverter (500 kW rated)", "unit": "Nos.", "qty_reference": 4, "total_cost_pkr": 0},
    {"item_no": 3, "description": "DC Cables", "unit": "m", "qty_reference": 37400, "total_cost_pkr": 0},
    {"item_no": 4, "description": "AC Cables", "unit": "m", "qty_reference": 350, "total_cost_pkr": 0},
    {"item_no": 5, "description": "Grounding Cable (PV)", "unit": "m", "qty_reference": 994, "total_cost_pkr": 0},
    {"item_no": 6, "description": "Grounding Cable (Structure)", "unit": "m", "qty_reference": 605, "total_cost_pkr": 0},
    {"item_no": 7, "description": "Grounding Cable (Inverter)", "unit": "m", "qty_reference": 64, "total_cost_pkr": 0},
    {"item_no": 8, "description": "DC Breakers", "unit": "Nos.", "qty_reference": 198, "total_cost_pkr": 0},
    {"item_no": 9, "description": "LV Switchgear", "unit": "Nos.", "qty_reference": 6, "total_cost_pkr": 0},
    {"item_no": 10, "description": "Cable Trays & Conduits", "unit": "Lot", "qty_reference": 6, "total_cost_pkr": 0},
    {"item_no": 11, "description": "Panel Mounting Structure", "unit": "Lot", "qty_reference": 1, "total_cost_pkr": 0},
    {"item_no": 12, "description": "Civil Work", "unit": "Lot", "qty_reference": 1, "total_cost_pkr": 0},
    {"item_no": 13, "description": "Water Cleaning System", "unit": "Lot", "qty_reference": 1, "total_cost_pkr": 0},
    {"item_no": 14, "description": "Transport Cost", "unit": "Lot", "qty_reference": 1, "total_cost_pkr": 0},
    {"item_no": 15, "description": "Commissioning + 3-month O&M", "unit": "Lot", "qty_reference": 1, "total_cost_pkr": 0},
    {"item_no": 16, "description": "Battery System", "unit": "MWh", "qty_reference": 8, "total_cost_pkr": 400_000_000},
    {"item_no": 17, "description": "Cables & Distribution Infrastructure", "unit": "Lot", "qty_reference": 1, "total_cost_pkr": 40_000_000},
])
default_boq_items.to_csv(DATA_DIR / "default_boq_items.csv", index=False)
print("Saved to:", (DATA_DIR / "default_boq_items.csv").resolve())
default_boq_items

Saved to: /home/claude/rural-electrification/data/default_boq_items.csv


,item_no,description,unit,qty_reference,total_cost_pkr
0,1,Solar Panels (590 W rated),W,2000000,164000000
1,2,Inverter (500 kW rated),Nos.,4,0
2,3,DC Cables,m,37400,0
3,4,AC Cables,m,350,0
4,5,Grounding Cable (PV),m,994,0
5,6,Grounding Cable (Structure),m,605,0
6,7,Grounding Cable (Inverter),m,64,0
7,8,DC Breakers,Nos.,198,0
8,9,LV Switchgear,Nos.,6,0
9,10,Cable Trays & Conduits,Lot,6,0


In [3]:
def compute_boq_total(boq_items_df: pd.DataFrame, ppeak_w: float, battery_capacity_kwh: float,
                       panel_unit_cost_per_w: float, battery_unit_cost_per_mwh: float) -> dict:
    """
    Recompute the BOQ grand total, replacing the 'Solar Panels' and 'Battery System' rows' stored total_cost_pkr
    with a live calculation from the current Ppeak / battery size — everything else uses the table's own values.
    Returns {"items": DataFrame with the live values substituted in, "capital_cost_pkr": grand total}.
    """
    items = boq_items_df.copy()  # copy(): don't mutate the caller's table
    panel_cost = panel_unit_cost_per_w * ppeak_w  # live link: panel cost tracks Step 4's Ppeak
    battery_cost = (battery_capacity_kwh / 1000) * battery_unit_cost_per_mwh  # live link: battery cost tracks Step 4's sized capacity (kWh -> MWh)

    items.loc[items["description"] == "Solar Panels (590 W rated)", "total_cost_pkr"] = panel_cost
    items.loc[items["description"] == "Battery System", "total_cost_pkr"] = battery_cost

    return {"items": items, "capital_cost_pkr": items["total_cost_pkr"].sum()}


# Validate against the workbook: same Ppeak (2 MW) and battery size (8,000 kWh, which happens to equal the
# workbook's own hand-typed 8 MWh) and the workbook's own per-unit rates (82 PKR/W panels, 50,000,000 PKR/MWh battery).
_boq_validation = compute_boq_total(default_boq_items, ppeak_w=2_000_000, battery_capacity_kwh=8000,
                                     panel_unit_cost_per_w=82, battery_unit_cost_per_mwh=50_000_000)
print(f"Computed capital cost: {_boq_validation['capital_cost_pkr']:,.0f} PKR  (workbook Annex-VI!H30: 604,000,000 PKR)")
assert _boq_validation["capital_cost_pkr"] == 604_000_000, "BOQ total does not match the workbook's cached grand total"
print("Match confirmed.")

Computed capital cost: 604,000,000 PKR  (workbook Annex-VI!H30: 604,000,000 PKR)
Match confirmed.


## 5.3 Land cost

Another loose end worth flagging: in the source workbook, the land-cost table (buy / private-lease / govt-lease, 3 options) never actually feeds into the capital-cost grand total — only the **govt-lease** figure separately reappears in the *operating*-cost section as one of the four Opex components. Buying land or leasing privately are computed for reference but included nowhere. Rather than silently keep that (possibly accidental) behavior, `land_cost_approach` below is an explicit, editable choice: `"govt_lease"` (default — matches the workbook), `"buy"` (one-time addition to Capex), `"private_lease"` (lump addition to Opex, same treatment as govt-lease), or `"none"`.

In [4]:
default_land_cost_options = pd.DataFrame([
    {"approach": "buy", "description": "Buying Land (one-time)", "acres": 5, "total_cost_pkr": 58_080_000},
    {"approach": "private_lease", "description": "Lease Private Land (30 years)", "acres": 5, "total_cost_pkr": 30_000_000},
    {"approach": "govt_lease", "description": "Lease Govt. Land (30 years)", "acres": 5, "total_cost_pkr": 4_000_000},
])
default_land_cost_options.to_csv(DATA_DIR / "default_land_cost_options.csv", index=False)
print("Saved to:", (DATA_DIR / "default_land_cost_options.csv").resolve())
default_land_cost_options

Saved to: /home/claude/rural-electrification/data/default_land_cost_options.csv


,approach,description,acres,total_cost_pkr
0,buy,Buying Land (one-time),5,58080000
1,private_lease,Lease Private Land (30 years),5,30000000
2,govt_lease,Lease Govt. Land (30 years),5,4000000


## 5.4 Operating cost present value (O&M)

Restated from the methodology doc §11.2: a year-1 cash flow (insurance + O&M), inflated every year, discounted back to present value, summed over 30 years.

In [5]:
def compute_om_present_value(capital_cost_pkr: float, insurance_pct: float, om_usd_per_kw: float, ppeak_kw: float,
                              usd_to_pkr_rate: float, inflation_rate: float, discount_rate: float, years: int = 30) -> dict:
    """
    Year-1 cash flow = (insurance_pct x capital cost) + (O&M $/kW/yr x system kW x USD->PKR rate), inflated at
    inflation_rate/yr and discounted back to present value at discount_rate/yr, summed over `years`.
    Returns {"cfo_year1_pkr", "present_value_pkr", "yearly_table": DataFrame[year, cash_flow_pkr, present_value_pkr]}.
    """
    insurance_yr1 = insurance_pct * capital_cost_pkr
    om_yr1 = om_usd_per_kw * ppeak_kw * usd_to_pkr_rate
    cfo_year1 = insurance_yr1 + om_yr1

    years_arr = np.arange(1, years + 1)  # arange(): 1, 2, ..., 30
    cash_flow = cfo_year1 * (1 + inflation_rate) ** (years_arr - 1)  # inflate the year-1 cash flow forward each year
    present_value = cash_flow / (1 + discount_rate) ** years_arr  # discount each year's cash flow back to today

    yearly_table = pd.DataFrame({"year": years_arr, "cash_flow_pkr": cash_flow, "present_value_pkr": present_value})
    return {"cfo_year1_pkr": cfo_year1, "present_value_pkr": present_value.sum(), "yearly_table": yearly_table}


# Validate against the workbook: same capital cost (604,000,000), insurance (0.5%), O&M ($1.8/kW/yr, 2000 kW),
# USD->PKR rate (280 — reverse-engineered from the workbook's own Cfo: matches exactly), inflation (8%), discount (10%).
_om_validation = compute_om_present_value(604_000_000, insurance_pct=0.005, om_usd_per_kw=1.8, ppeak_kw=2000,
                                           usd_to_pkr_rate=280, inflation_rate=0.08, discount_rate=0.10)
print(f"Year-1 cash flow: {_om_validation['cfo_year1_pkr']:,.0f} PKR  (workbook Annex-VI!N39: 4,028,000 PKR)")
print(f"O&M+Insurance PV:  {_om_validation['present_value_pkr']:,.2f} PKR  (workbook Annex-VI!P72: 85,257,391.43 PKR)")
assert _om_validation["cfo_year1_pkr"] == 4_028_000, "Year-1 cash flow does not match the workbook"
assert abs(_om_validation["present_value_pkr"] - 85_257_391.43) < 1, "O&M present value does not match the workbook"
print("Match confirmed.")

Year-1 cash flow: 4,028,000 PKR  (workbook Annex-VI!N39: 4,028,000 PKR)
O&M+Insurance PV:  85,257,391.43 PKR  (workbook Annex-VI!P72: 85,257,391.43 PKR)
Match confirmed.


## 5.5 Total cost and LCOE

In [6]:
def compute_total_opex(om_present_value_pkr: float, battery_capex_pkr: float, capital_cost_pkr: float,
                        inverter_replacement_pct: float, land_cost_pkr: float) -> dict:
    """
    Total 30-year operating cost = O&M/insurance present value + a one-time battery replacement (year 12, full
    replacement cost, undiscounted — matching the workbook's own treatment) + inverter replacement (a percentage
    of the non-battery capital cost) + the chosen land-cost line. Returns the total plus a labeled breakdown.
    """
    inverter_replacement_pkr = inverter_replacement_pct * (capital_cost_pkr - battery_capex_pkr)
    breakdown = {
        "om_insurance_present_value_pkr": om_present_value_pkr,
        "battery_replacement_pkr": battery_capex_pkr,
        "inverter_replacement_pkr": inverter_replacement_pkr,
        "land_cost_pkr": land_cost_pkr,
    }
    return {"total_opex_pkr": sum(breakdown.values()), "breakdown": breakdown}


def compute_lcoe(capital_cost_pkr: float, total_opex_pkr: float, annual_energy_wh: float, years: int = 30) -> dict:
    """Total cost (Capex+Opex) / Total energy over `years` years, in PKR/kWh. Pass annual PV GENERATION (not demand) per project decision #3."""
    total_cost_pkr = capital_cost_pkr + total_opex_pkr
    total_energy_kwh = (annual_energy_wh / 1000) * years
    return {"total_cost_pkr": total_cost_pkr, "total_energy_kwh": total_energy_kwh, "lcoe_pkr_per_kwh": total_cost_pkr / total_energy_kwh}


# Validate against the workbook: battery replacement = the same 400,000,000 PKR hand-typed battery capex, inverter
# replacement = 15% of (capital - battery), land = the govt-lease figure (4,000,000) — and, deliberately reproducing
# the workbook's mislabeling for this validation step only, the DEMAND sum (not generation) as the energy denominator.
_opex_validation = compute_total_opex(_om_validation["present_value_pkr"], battery_capex_pkr=400_000_000,
                                       capital_cost_pkr=604_000_000, inverter_replacement_pct=0.15, land_cost_pkr=4_000_000)
print(f"Total Opex: {_opex_validation['total_opex_pkr']:,.2f} PKR  (workbook Annex-VI!H45: 519,857,391.43 PKR)")
assert abs(_opex_validation["total_opex_pkr"] - 519_857_391.43) < 1, "Total Opex does not match the workbook"

# NOTE: this validation step uses the WORKBOOK's own as-implemented demand figure (3,472,586,398 Wh) directly, not the
# `annual_demand_wh` variable loaded above (which is this PROJECT's corrected, slightly different demand from Steps 2-3) —
# this cell is purely about matching the workbook's cached LCOE number.
_lcoe_validation_demand = compute_lcoe(604_000_000, _opex_validation["total_opex_pkr"], annual_energy_wh=3_472_586_398)
print(f"Total Cost: {_lcoe_validation_demand['total_cost_pkr']:,.2f} PKR  (workbook: 1,123,857,391.43 PKR)")
print(f"LCOE (demand-based, validation only): {_lcoe_validation_demand['lcoe_pkr_per_kwh']:.4f} PKR/kWh  (workbook: 10.7879 PKR/kWh)")
assert abs(_lcoe_validation_demand["lcoe_pkr_per_kwh"] - 10.7879) < 0.001, "LCOE does not match the workbook"
print("Match confirmed — BOQ, Opex, and LCOE all reproduce the workbook exactly.")

# Now apply decision #3: LCOE = Total Cost / Total GENERATION, not demand.
_lcoe_validation_generation = compute_lcoe(604_000_000, _opex_validation["total_opex_pkr"], annual_energy_wh=annual_egen_wh)
print()
print(f"LCOE (generation-based, per decision #3): {_lcoe_validation_generation['lcoe_pkr_per_kwh']:.4f} PKR/kWh")
print(f"  -> {(_lcoe_validation_generation['lcoe_pkr_per_kwh']/_lcoe_validation_demand['lcoe_pkr_per_kwh'] - 1)*100:+.1f}% vs. the demand-based (mislabeled) figure the workbook actually computed —")
print("     generation exceeds demand by design (the plant is sized with headroom/curtailment), so spreading the same total cost over more kWh lowers LCOE.")

Total Opex: 519,857,391.43 PKR  (workbook Annex-VI!H45: 519,857,391.43 PKR)
Total Cost: 1,123,857,391.43 PKR  (workbook: 1,123,857,391.43 PKR)
LCOE (demand-based, validation only): 10.7879 PKR/kWh  (workbook: 10.7879 PKR/kWh)
Match confirmed — BOQ, Opex, and LCOE all reproduce the workbook exactly.

LCOE (generation-based, per decision #3): 9.7562 PKR/kWh
  -> -9.6% vs. the demand-based (mislabeled) figure the workbook actually computed —
     generation exceeds demand by design (the plant is sized with headroom/curtailment), so spreading the same total cost over more kWh lowers LCOE.


## 5.6 Cost parameters (all editable) and the currency toggle

Every scalar input from §5.2-5.5 in one editable table, same pattern as Step 4's PV parameters — plus the lump-sum override and land-cost-approach choice discussed above.

In [7]:
default_cost_parameters = pd.DataFrame([
    {"parameter": "panel_unit_cost_pkr_per_w", "value": 82, "unit": "PKR/W", "description": "Solar panel cost per watt — drives the BOQ's live-linked Panels line"},
    {"parameter": "battery_unit_cost_pkr_per_mwh", "value": 50_000_000, "unit": "PKR/MWh", "description": "Battery cost per MWh — drives the BOQ's live-linked Battery System line"},
    {"parameter": "land_cost_approach", "value": "govt_lease", "unit": "buy | private_lease | govt_lease | none", "description": "Which land-cost figure (if any) is included in the totals"},
    {"parameter": "insurance_pct_per_year", "value": 0.005, "unit": "fraction/year", "description": "Insurance cost, as a fraction of capital cost per year"},
    {"parameter": "om_usd_per_kw_per_year", "value": 1.8, "unit": "USD/kW/year", "description": "Operations & maintenance cost per kW of Ppeak per year"},
    {"parameter": "usd_to_pkr_rate", "value": 280, "unit": "PKR/USD", "description": "Exchange rate used to convert the O&M cost from USD to PKR"},
    {"parameter": "eur_to_pkr_rate", "value": 330, "unit": "PKR/EUR", "description": "Exchange rate used for the EUR currency-toggle view"},
    {"parameter": "inflation_rate", "value": 0.08, "unit": "fraction/year", "description": "Annual inflation applied to the O&M cash flow"},
    {"parameter": "discount_rate", "value": 0.10, "unit": "fraction/year", "description": "Annual discount rate used to present-value the O&M cash flow"},
    {"parameter": "inverter_replacement_pct", "value": 0.15, "unit": "fraction", "description": "Inverter replacement cost, as a fraction of (capital cost minus battery cost)"},
    {"parameter": "project_lifetime_years", "value": 30, "unit": "years", "description": "Project lifetime used for the O&M cash-flow horizon and the LCOE energy denominator"},
    {"parameter": "use_lump_sum_capex", "value": 0, "unit": "0 or 1 (boolean)", "description": "If 1, override the itemized BOQ total with lump_sum_capex_pkr below"},
    {"parameter": "lump_sum_capex_pkr", "value": 604_000_000, "unit": "PKR", "description": "Used only when use_lump_sum_capex=1 — a single all-in capital cost figure"},
])
default_cost_parameters.to_csv(DATA_DIR / "default_cost_parameters.csv", index=False)
print("Saved to:", (DATA_DIR / "default_cost_parameters.csv").resolve())
default_cost_parameters

Saved to: /home/claude/rural-electrification/data/default_cost_parameters.csv


,parameter,value,unit,description
0,panel_unit_cost_pkr_per_w,82,PKR/W,Solar panel cost per watt — drives the BOQ's l...
1,battery_unit_cost_pkr_per_mwh,50000000,PKR/MWh,Battery cost per MWh — drives the BOQ's live-l...
2,land_cost_approach,govt_lease,buy | private_lease | govt_lease | none,Which land-cost figure (if any) is included in...
3,insurance_pct_per_year,0.005,fraction/year,"Insurance cost, as a fraction of capital cost ..."
4,om_usd_per_kw_per_year,1.8,USD/kW/year,Operations & maintenance cost per kW of Ppeak ...
5,usd_to_pkr_rate,280,PKR/USD,Exchange rate used to convert the O&M cost fro...
6,eur_to_pkr_rate,330,PKR/EUR,Exchange rate used for the EUR currency-toggle...
7,inflation_rate,0.08,fraction/year,Annual inflation applied to the O&M cash flow
8,discount_rate,0.1,fraction/year,Annual discount rate used to present-value the...
9,inverter_replacement_pct,0.15,fraction,"Inverter replacement cost, as a fraction of (c..."


In [8]:
def get_param(parameters_df: pd.DataFrame, name: str):
    """Look up one parameter's value by name from a parameters table (default or user-edited) — same helper as Step 4's."""
    return parameters_df.loc[parameters_df["parameter"] == name, "value"].iloc[0]


def convert_currency(pkr_value: float, currency: str, eur_to_pkr_rate: float, usd_to_pkr_rate: float) -> float:
    """Convert a PKR amount to PKR / EUR / USD for display. currency is one of 'PKR', 'EUR', 'USD'."""
    if currency == "PKR":
        return pkr_value
    elif currency == "EUR":
        return pkr_value / eur_to_pkr_rate
    elif currency == "USD":
        return pkr_value / usd_to_pkr_rate
    raise ValueError(f"Unknown currency '{currency}'. Choose one of: PKR, EUR, USD")


# Demo: the same total cost, shown in all three currencies.
_eur_rate = get_param(default_cost_parameters, "eur_to_pkr_rate")
_usd_rate = get_param(default_cost_parameters, "usd_to_pkr_rate")
for currency in ["PKR", "EUR", "USD"]:
    print(f"Total Cost (validation figure): {convert_currency(1_123_857_391.43, currency, _eur_rate, _usd_rate):,.2f} {currency}")

Total Cost (validation figure): 1,123,857,391.43 PKR
Total Cost (validation figure): 3,405,628.46 EUR
Total Cost (validation figure): 4,013,776.40 USD


## 5.7 The full pipeline, using this project's own numbers

Everything above tied together into one function, using this project's own generation figure (decision #3) as the LCOE denominator, the live-linked BOQ (Step 4's actual Ppeak/battery size), the chosen land-cost approach, and — demonstrated with both settings — the lump-sum override.

In [9]:
def run_cost_lcoe_pipeline(boq_items_df: pd.DataFrame, land_cost_options_df: pd.DataFrame, cost_parameters_df: pd.DataFrame,
                            ppeak_w: float, battery_capacity_kwh: float, annual_egen_wh: float) -> dict:
    """
    The full Step 5 pipeline: BOQ -> (optional lump-sum override) -> O&M present value -> total Opex (incl. the
    chosen land-cost line) -> LCOE, using annual PV GENERATION as the energy denominator (decision #3).
    Returns a dict with every intermediate result, so the app can show a full breakdown, not just the final number.
    """
    panel_cost = get_param(cost_parameters_df, "panel_unit_cost_pkr_per_w")
    battery_cost = get_param(cost_parameters_df, "battery_unit_cost_pkr_per_mwh")
    boq_result = compute_boq_total(boq_items_df, ppeak_w, battery_capacity_kwh, panel_cost, battery_cost)
    battery_capex_pkr = (battery_capacity_kwh / 1000) * battery_cost  # needed again below for the inverter-replacement calc

    use_lump_sum = bool(get_param(cost_parameters_df, "use_lump_sum_capex"))
    capital_cost_pkr = get_param(cost_parameters_df, "lump_sum_capex_pkr") if use_lump_sum else boq_result["capital_cost_pkr"]

    land_approach = get_param(cost_parameters_df, "land_cost_approach")
    if land_approach == "none":
        land_cost_pkr = 0
    else:
        land_cost_pkr = land_cost_options_df.loc[land_cost_options_df["approach"] == land_approach, "total_cost_pkr"].iloc[0]
    # "buy" is a one-time addition to Capex (it's not a recurring cost like the two lease options); leases stay in Opex.
    if land_approach == "buy":
        capital_cost_pkr += land_cost_pkr
        land_cost_for_opex = 0
    else:
        land_cost_for_opex = land_cost_pkr

    ppeak_kw = ppeak_w / 1000
    om_result = compute_om_present_value(
        capital_cost_pkr, get_param(cost_parameters_df, "insurance_pct_per_year"), get_param(cost_parameters_df, "om_usd_per_kw_per_year"),
        ppeak_kw, get_param(cost_parameters_df, "usd_to_pkr_rate"), get_param(cost_parameters_df, "inflation_rate"),
        get_param(cost_parameters_df, "discount_rate"), years=int(get_param(cost_parameters_df, "project_lifetime_years")),
    )
    opex_result = compute_total_opex(om_result["present_value_pkr"], battery_capex_pkr, capital_cost_pkr,
                                      get_param(cost_parameters_df, "inverter_replacement_pct"), land_cost_for_opex)
    lcoe_result = compute_lcoe(capital_cost_pkr, opex_result["total_opex_pkr"], annual_egen_wh,
                                years=int(get_param(cost_parameters_df, "project_lifetime_years")))

    return {"boq": boq_result, "capital_cost_pkr": capital_cost_pkr, "om": om_result, "opex": opex_result, "lcoe": lcoe_result}


# Run with the itemized BOQ (use_lump_sum_capex = 0, the default)
result_itemized = run_cost_lcoe_pipeline(default_boq_items, default_land_cost_options, default_cost_parameters,
                                          ppeak_w, battery_capacity_kwh, annual_egen_wh)
print("=== Itemized BOQ ===")
print(f"Capital cost: {result_itemized['capital_cost_pkr']:,.0f} PKR")
print(f"Total Opex:   {result_itemized['opex']['total_opex_pkr']:,.2f} PKR")
print(f"Total Cost:   {result_itemized['lcoe']['total_cost_pkr']:,.2f} PKR")
print(f"LCOE:         {result_itemized['lcoe']['lcoe_pkr_per_kwh']:.4f} PKR/kWh")

# Demonstrate the lump-sum override: same everything, but capital cost overridden to a single round number.
_lump_sum_params = default_cost_parameters.copy()
_lump_sum_params.loc[_lump_sum_params["parameter"] == "use_lump_sum_capex", "value"] = 1
_lump_sum_params.loc[_lump_sum_params["parameter"] == "lump_sum_capex_pkr", "value"] = 650_000_000  # a different, illustrative all-in figure
result_lump_sum = run_cost_lcoe_pipeline(default_boq_items, default_land_cost_options, _lump_sum_params,
                                          ppeak_w, battery_capacity_kwh, annual_egen_wh)
print()
print("=== Lump-sum override (650,000,000 PKR) ===")
print(f"Capital cost: {result_lump_sum['capital_cost_pkr']:,.0f} PKR")
print(f"LCOE:         {result_lump_sum['lcoe']['lcoe_pkr_per_kwh']:.4f} PKR/kWh")
assert result_lump_sum["capital_cost_pkr"] == 650_000_000, "Lump-sum override did not take effect"
print("Lump-sum override confirmed working — the itemized BOQ is still computed either way (for reference), just not used for the total when the override is on.")

=== Itemized BOQ ===
Capital cost: 604,000,000 PKR
Total Opex:   519,857,391.43 PKR
Total Cost:   1,123,857,391.43 PKR
LCOE:         9.7562 PKR/kWh

=== Lump-sum override (650,000,000 PKR) ===
Capital cost: 650,000,000 PKR
LCOE:         10.2576 PKR/kWh
Lump-sum override confirmed working — the itemized BOQ is still computed either way (for reference), just not used for the total when the override is on.


## 5.8 Excel upload/download for the BOQ and cost parameters

In [10]:
def export_cost_template(boq_items_df: pd.DataFrame, land_cost_options_df: pd.DataFrame, cost_parameters_df: pd.DataFrame, output_path) -> None:
    """Write the BOQ, land-cost options, and cost parameters to one formatted .xlsx template."""
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "bg_color": "#2a78d6", "font_color": "white", "border": 1})

        instructions = pd.DataFrame({"Instructions": [
            "BOQ Items: edit total_cost_pkr for any line. The 'Solar Panels' and 'Battery System' rows are recomputed live from Step 4 and the unit-cost parameters below, so their values here are for reference only and will be overridden.",
            "Land Cost Options: edit total_cost_pkr for any of the three approaches; which one is actually used is chosen by the land_cost_approach parameter.",
            "Cost Parameters: edit the 'value' column only — do not add/remove/rename rows.",
            "Save the file, then re-upload it in the app to use these values instead of the defaults.",
        ]})
        instructions.to_excel(writer, sheet_name="Instructions", index=False)
        boq_items_df.to_excel(writer, sheet_name="BOQ Items", index=False)
        land_cost_options_df.to_excel(writer, sheet_name="Land Cost Options", index=False)
        cost_parameters_df.to_excel(writer, sheet_name="Cost Parameters", index=False)

        for sheet_name, df in [("Instructions", instructions), ("BOQ Items", boq_items_df),
                                ("Land Cost Options", land_cost_options_df), ("Cost Parameters", cost_parameters_df)]:
            ws = writer.sheets[sheet_name]
            for col_idx, col_name in enumerate(df.columns):
                ws.write(0, col_idx, col_name, header_fmt)
                ws.set_column(col_idx, col_idx, max(14, len(str(col_name)) + 2))


def import_cost_template(input_path) -> dict:
    """Read a workbook in the export_cost_template() shape back into the three tables, with validation."""
    errors = []
    sheets = pd.read_excel(input_path, sheet_name=None)
    required_sheets = ["BOQ Items", "Land Cost Options", "Cost Parameters"]
    missing_sheets = [s for s in required_sheets if s not in sheets]
    if missing_sheets:
        return {"boq_items": None, "land_cost_options": None, "cost_parameters": None, "errors": [f"Missing required sheet(s): {missing_sheets}"]}

    boq_items, land_cost_options, cost_parameters = sheets["BOQ Items"], sheets["Land Cost Options"], sheets["Cost Parameters"]

    if set(boq_items.get("item_no", [])) != set(default_boq_items["item_no"]):
        errors.append("'BOQ Items' must contain exactly the same 17 item_no rows as the default")
    if (boq_items.get("total_cost_pkr", pd.Series(dtype=float)) < 0).any():
        errors.append("BOQ total_cost_pkr values must not be negative")
    if set(land_cost_options.get("approach", [])) != set(default_land_cost_options["approach"]):
        errors.append("'Land Cost Options' must contain exactly the same 3 approach rows as the default")
    if set(cost_parameters.get("parameter", [])) != set(default_cost_parameters["parameter"]):
        errors.append("'Cost Parameters' must contain exactly the same parameter rows as the default")
    valid_land_approaches = {"buy", "private_lease", "govt_lease", "none"}
    _land_choice = cost_parameters.loc[cost_parameters["parameter"] == "land_cost_approach", "value"]
    if not _land_choice.empty and _land_choice.iloc[0] not in valid_land_approaches:
        errors.append(f"land_cost_approach must be one of {sorted(valid_land_approaches)}")

    if errors:
        return {"boq_items": None, "land_cost_options": None, "cost_parameters": None, "errors": errors}
    return {"boq_items": boq_items, "land_cost_options": land_cost_options, "cost_parameters": cost_parameters, "errors": []}


export_cost_template(default_boq_items, default_land_cost_options, default_cost_parameters, DATA_DIR / "Cost_BOQ_Template.xlsx")
print("Saved to:", (DATA_DIR / "Cost_BOQ_Template.xlsx").resolve())

_imported = import_cost_template(DATA_DIR / "Cost_BOQ_Template.xlsx")
assert _imported["errors"] == [], f"Round-trip import reported errors: {_imported['errors']}"
pd.testing.assert_frame_equal(_imported["boq_items"].reset_index(drop=True), default_boq_items.reset_index(drop=True))
pd.testing.assert_frame_equal(_imported["land_cost_options"].reset_index(drop=True), default_land_cost_options.reset_index(drop=True))
pd.testing.assert_frame_equal(_imported["cost_parameters"].reset_index(drop=True), default_cost_parameters.reset_index(drop=True))
print("Round-trip OK: exported defaults re-imported with zero errors and an exact match on all three tables.")

Saved to: /home/claude/rural-electrification/data/Cost_BOQ_Template.xlsx


Round-trip OK: exported defaults re-imported with zero errors and an exact match on all three tables.


## 5.9 Saving results

In [11]:
cost_lcoe_results = pd.DataFrame([
    {"result": "capital_cost_pkr", "value": result_itemized["capital_cost_pkr"], "unit": "PKR"},
    {"result": "total_opex_pkr", "value": result_itemized["opex"]["total_opex_pkr"], "unit": "PKR"},
    {"result": "total_cost_pkr", "value": result_itemized["lcoe"]["total_cost_pkr"], "unit": "PKR"},
    {"result": "total_energy_kwh_30yr", "value": result_itemized["lcoe"]["total_energy_kwh"], "unit": "kWh"},
    {"result": "lcoe_pkr_per_kwh", "value": result_itemized["lcoe"]["lcoe_pkr_per_kwh"], "unit": "PKR/kWh"},
])
cost_lcoe_results.to_csv(DATA_DIR / "cost_lcoe_results_2026.csv", index=False)
print("Saved to:", (DATA_DIR / "cost_lcoe_results_2026.csv").resolve())
cost_lcoe_results

Saved to: /home/claude/rural-electrification/data/cost_lcoe_results_2026.csv


,result,value,unit
0,capital_cost_pkr,6.040000e+08,PKR
1,total_opex_pkr,5.198574e+08,PKR
2,total_cost_pkr,1.123857e+09,PKR
3,total_energy_kwh_30yr,1.151947e+08,kWh
4,lcoe_pkr_per_kwh,9.756155e+00,PKR/kWh


## 5.10 Summary

- **BOQ**: 17-line itemized bill of quantities, validated to the exact PKR against the workbook (604,000,000 PKR). Flagged plainly: 14 of the 17 lines have zero cost in the source (an unfinished BOQ, not a design choice) — every line is a normal editable value. The **Solar Panels** and **Battery System** lines are now genuinely live-computed from Step 4's Ppeak and battery size (the methodology doc's own flagged fix), not hand-typed.
- **Land cost**: the workbook's govt-lease figure is the only one that ever reached a total (buy/private-lease were reference-only, unused) — made an explicit, editable `land_cost_approach` choice instead of leaving that silent.
- **O&M present value**: validated to the exact PKR against the workbook (85,257,391.43 PKR), using an inflate-then-discount 30-year cash flow.
- **LCOE**: validated against the workbook's own (demand-based, mislabeled) figure (10.7879 PKR/kWh) — then switched to **generation-based**, per decision #3: **9.7562 PKR/kWh**, about 9.6% lower, because generation exceeds demand by design.
- **Currency toggle** (`convert_currency()`): PKR / EUR / USD, using two separate, editable exchange rates (the O&M calc's own implied 280 PKR/USD, and the workbook's 330 PKR/EUR).
- **Lump-sum override**: `use_lump_sum_capex` + `lump_sum_capex_pkr` bypass the itemized BOQ entirely when set, demonstrated and confirmed working.
- Everything editable via `Cost_BOQ_Template.xlsx` (BOQ items, land cost options, cost parameters), same validated round-trip pattern as every other dataset.
- New files in `data/`: `default_boq_items.csv`, `default_land_cost_options.csv`, `default_cost_parameters.csv`, `Cost_BOQ_Template.xlsx`, `cost_lcoe_results_2026.csv`.

**Step 5 (Cost / BOQ / LCOE) is now done.** Next: Step 6 — the Summary Table, pulling together every prior step's results (demand, connected load, system size, battery size, cost, LCOE) plus a defined ROI/payback metric (none exists in the source workbook — flagged back in the project log).